# GNN Model Report 

## 1. Model Overview

We implement a **Graph Neural Network (GNN)** to predict startup success by directly leveraging the venture capital investment network.

In this framework:
- Each **company is a node**
- Connections between companies are defined through **shared investors**
- The model learns by aggregating information from neighboring companies in the network

We use a **GraphSAGE architecture**, which updates each company’s representation by combining its own features with information from its connected peers.

---

## 2. Data and Graph Construction

### Dataset
- Total labeled companies: **6,593**
- Target variable: **`is_success`** (binary outcome)

---

### Graph Structure
- Nodes: companies
- Edges: two companies are connected if they share at least one investor

This results in:
- ~510,000 undirected edges
- A dense network capturing co-investment relationships

---

### Feature Sets

#### **GNN V1: Network + Tabular Features**
Includes:
- company-level attributes
- precomputed network statistics (e.g., centrality, degree)

Excludes leakage variables:
- `funding_total_usd`  
- `log_funding`  
- `num_funding_rounds`  
- `company_age_months`  

---

#### **GNN V2: Network + Tabular + Education/Job Features**
Extends V1 by adding:
- founder education signals (e.g., elite institutions)
- employment background (e.g., top companies)

These features aim to capture human capital effects.

---

## 3. Model Architecture and Training

### Architecture
- 2-layer GraphSAGE
- Hidden dimension: 64
- ReLU activation + dropout

---

### Training Setup
- Loss: weighted cross-entropy (to address class imbalance)
- Data split:
  - 64% training
  - 16% validation
  - 20% test
- Early stopping based on validation ROC-AUC

---

## 4. Results

### Performance Summary

| Model   | Val ROC-AUC | Val PR-AUC | Test ROC-AUC | Test PR-AUC |
|--------|------------|-----------|-------------|------------|
| GNN V1 | 0.8398     | 0.6233    | **0.8015**  | 0.5141     |
| GNN V2 | 0.8391     | 0.6174    | 0.7960      | 0.5137     |

---

### Classification Behavior

- Strong performance on the majority class (non-success)
- Moderate recall for successful companies (~0.68)
- Lower precision for successful companies (~0.35)

This reflects:
- class imbalance in the dataset  
- inherent uncertainty in predicting startup outcomes  

---

## 5. Key Findings

### 1. Network structure provides strong predictive signal

The GNN achieves high predictive performance, indicating that:
- relationships between companies (via shared investors)
- and their positions within the investment network  

are highly informative for predicting outcomes.

---

### 2. Information propagates through the network

The model captures:
- influence from connected investors  
- similarity between co-invested companies  
- higher-order relationships beyond direct connections  

This allows each company’s prediction to incorporate **multi-hop network effects**.

---

### 3. Education and job features provide limited additional value

Adding education and employment features does not improve performance.

Possible reasons:
- these features are sparse or incomplete  
- their signal is weak relative to network structure  
- their effects may already be indirectly reflected through the network  

---

## 6. Interpretation

The GNN learns latent representations (embeddings) of companies that encode:
- investor quality  
- network position  
- shared investment patterns  

These embeddings summarize both:
- local neighborhood information  
- broader structural context  

As a result, the model is able to detect patterns such as:
- clusters of successful companies  
- strong investor ecosystems  
- indirect connections to high-quality networks  

---

## 7. Limitations

- The graph is relatively dense, which may introduce noise from large investors  
- Node features already include some network-derived statistics, which may overlap with graph information  
- Model interpretability is limited compared to simpler approaches  

---

## 8. Conclusion

> A Graph Neural Network can effectively model startup success by directly leveraging the structure of the venture capital investment network.

Key takeaways:
- Network relationships are highly informative  
- GNN successfully captures both local and global structure  
- Additional social features (education/job) provide limited incremental value  

---

## 9. Future Work

Potential improvements include:
- refining graph construction (e.g., weighting edges by investment strength)  
- removing engineered network features to isolate pure graph learning  
- extending to heterogeneous graphs (companies, investors, individuals)  
- exploring attention-based GNN models for improved interpretability  

In [6]:
pip install torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 3.0 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.9 MB/s eta 0:00:002.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.9/679.9 kB 1.3 MB/s eta 0:00:002.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchvision]━━━━━━ 2/3 [torchvision]
Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install torch-geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torch-geometric]1/2 [torch-geometric]
Note: you may need to restart the kernel to use updated packages.


In [10]:
import copy
import itertools
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import to_undirected, remove_self_loops


In [11]:
# ============================================================
# 1. File paths
# ============================================================
FEATURE_PATH = "feature_matrix.csv"
EDU_JOB_PATH = "edu_job_features.csv"
PORTFOLIO_PATH = "portfolio_edges.csv"


# ============================================================
# 2. Load datasets
# ============================================================
base = pd.read_csv(FEATURE_PATH)
edu_job = pd.read_csv(EDU_JOB_PATH)
portfolio = pd.read_csv(PORTFOLIO_PATH)

print("Loaded files:")
print("feature_matrix:", base.shape)
print("edu_job_features:", edu_job.shape)
print("portfolio_edges:", portfolio.shape)



Loaded files:
feature_matrix: (6704, 37)
edu_job_features: (6704, 15)
portfolio_edges: (461543, 6)


In [12]:

# ============================================================
# 3. Merge base + edu/job
# ============================================================
full = base.merge(edu_job, on="company_uuid", how="left")

# Keep only labeled rows
full = full[full["is_success"].notna()].copy()
full["is_success"] = full["is_success"].astype(int)

print("\nAfter keeping labeled companies only:")
print(full.shape)
print(full["is_success"].value_counts(dropna=False))



After keeping labeled companies only:
(6593, 51)
is_success
0    5477
1    1116
Name: count, dtype: int64


In [13]:
# ============================================================
# 4. Define feature groups
# ============================================================
# Exclude leakage features based on your report
leakage_cols = [
    "funding_total_usd",
    "log_funding",
    "num_funding_rounds",
    "company_age_months",
]

id_target_cols = ["company_uuid", "is_success"]

# V1 features = base columns only, excluding leakage + id/target
base_cols = base.columns.tolist()
v1_feature_cols = [
    c for c in base_cols
    if c not in leakage_cols + id_target_cols
]

# V2 features = merged full columns, excluding leakage + id/target
full_cols = full.columns.tolist()
v2_feature_cols = [
    c for c in full_cols
    if c not in leakage_cols + id_target_cols
]

print("\nFeature counts:")
print("V1 feature count:", len(v1_feature_cols))
print("V2 feature count:", len(v2_feature_cols))



Feature counts:
V1 feature count: 31
V2 feature count: 45

Number of labeled companies: 6593


In [14]:
# ============================================================
# 5. Align company order and create labels
# ============================================================
# We will use the row order of 'full' as the node order
company_ids = full["company_uuid"].tolist()
company_to_idx = {cid: i for i, cid in enumerate(company_ids)}

y = full["is_success"].values.astype(int)

print("\nNumber of labeled companies:", len(company_ids))




Number of labeled companies: 6593


In [15]:

# ============================================================
# 6. Build feature matrices
# ============================================================
# V1 uses only base columns; need to align to full's company order
base_labeled = base[base["company_uuid"].isin(company_ids)].copy()
base_labeled = base_labeled.set_index("company_uuid").loc[company_ids].reset_index()

# V2 uses full directly
full_aligned = full.set_index("company_uuid").loc[company_ids].reset_index()

def preprocess_features(df, feature_cols):
    """
    Median imputation + standard scaling
    Returns numpy array X
    """
    X = df[feature_cols].copy()

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    return X_scaled, imputer, scaler

X_v1, imputer_v1, scaler_v1 = preprocess_features(base_labeled, v1_feature_cols)
X_v2, imputer_v2, scaler_v2 = preprocess_features(full_aligned, v2_feature_cols)

print("\nProcessed feature matrices:")
print("X_v1:", X_v1.shape)
print("X_v2:", X_v2.shape)




Processed feature matrices:
X_v1: (6593, 31)
X_v2: (6593, 45)


In [16]:
# ============================================================
# 7. Build company-company graph from shared investors
# ============================================================
# portfolio_edges.csv has:
# - vc_uuid
# - portfolio_company_uuid
#
# We connect two companies if they share at least one investor.

portfolio = portfolio.copy()

# Keep only companies that appear in our labeled set
valid_companies = set(company_ids)
portfolio = portfolio[portfolio["portfolio_company_uuid"].isin(valid_companies)].copy()

print("\nFiltered portfolio edges shape:", portfolio.shape)

# Group by investor
investor_to_companies = (
    portfolio.groupby("vc_uuid")["portfolio_company_uuid"]
    .apply(lambda x: list(set(x)))   # remove duplicates within investor
    .to_dict()
)

print("Number of investors with labeled portfolio companies:", len(investor_to_companies))


def build_shared_investor_edge_index(investor_to_companies, company_to_idx, max_pairs_per_investor=None):
    """
    Build undirected company-company graph:
    connect companies that share an investor.

    max_pairs_per_investor:
        optional cap to avoid extremely dense graphs from huge investors.
        None = no cap.
    """
    edge_set = set()

    for investor, company_list in investor_to_companies.items():
        if len(company_list) < 2:
            continue

        # All unordered company pairs for this investor
        pairs = list(itertools.combinations(company_list, 2))

        # Optional cap if graph gets too dense
        if max_pairs_per_investor is not None and len(pairs) > max_pairs_per_investor:
            pairs = pairs[:max_pairs_per_investor]

        for c1, c2 in pairs:
            i = company_to_idx[c1]
            j = company_to_idx[c2]

            if i != j:
                # store ordered tuple to avoid duplicates
                edge_set.add((min(i, j), max(i, j)))

    edge_list = list(edge_set)

    src = [e[0] for e in edge_list]
    dst = [e[1] for e in edge_list]

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_index = to_undirected(edge_index)
    edge_index, _ = remove_self_loops(edge_index)

    return edge_index

# Build real graph
# If graph becomes too large / slow, set max_pairs_per_investor=20000 or smaller
edge_index = build_shared_investor_edge_index(
    investor_to_companies=investor_to_companies,
    company_to_idx=company_to_idx,
    max_pairs_per_investor=None
)

print("\nGraph built:")
print("edge_index shape:", edge_index.shape)
print("num undirected edges:", edge_index.shape[1] // 2)



Filtered portfolio edges shape: (38607, 6)
Number of investors with labeled portfolio companies: 8504

Graph built:
edge_index shape: torch.Size([2, 1021972])
num undirected edges: 510986


In [17]:
# ============================================================
# 8. Train / validation / test split
# ============================================================
n = len(company_ids)
all_idx = np.arange(n)

train_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    stratify=y,
    random_state=42
)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.20,   # 0.20 of train => 64/16/20 split overall
    stratify=y[train_idx],
    random_state=42
)

def make_mask(indices, n):
    mask = torch.zeros(n, dtype=torch.bool)
    mask[indices] = True
    return mask

train_mask = make_mask(train_idx, n)
val_mask = make_mask(val_idx, n)
test_mask = make_mask(test_idx, n)

print("\nSplit sizes:")
print("Train:", train_mask.sum().item())
print("Val  :", val_mask.sum().item())
print("Test :", test_mask.sum().item())




Split sizes:
Train: 4219
Val  : 1055
Test : 1319


In [18]:
# ============================================================
# 9. Build PyG Data objects
# ============================================================
def make_data_object(X, y, edge_index, train_mask, val_mask, test_mask):
    data = Data(
        x=torch.tensor(X, dtype=torch.float32),
        edge_index=edge_index,
        y=torch.tensor(y, dtype=torch.long)
    )
    data.train_mask = train_mask
    data.val_mask = val_mask
    data.test_mask = test_mask
    return data

data_v1 = make_data_object(X_v1, y, edge_index, train_mask, val_mask, test_mask)
data_v2 = make_data_object(X_v2, y, edge_index, train_mask, val_mask, test_mask)




In [19]:
# ============================================================
# 10. Define GraphSAGE model
# ============================================================
class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin = nn.Linear(hidden_channels, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        out = self.lin(x)
        return out


In [20]:

# ============================================================
# 11. Evaluation helpers
# ============================================================
def get_probabilities(logits):
    probs = F.softmax(logits, dim=1)[:, 1]
    return probs

def evaluate_split(logits, y_true, mask, threshold=0.5):
    probs = get_probabilities(logits).detach().cpu().numpy()
    y_np = y_true.detach().cpu().numpy()
    mask_np = mask.detach().cpu().numpy()

    y_eval = y_np[mask_np]
    p_eval = probs[mask_np]
    pred_eval = (p_eval >= threshold).astype(int)

    roc = roc_auc_score(y_eval, p_eval)
    pr = average_precision_score(y_eval, p_eval)

    return {
        "roc_auc": roc,
        "pr_auc": pr,
        "y_true": y_eval,
        "y_pred": pred_eval,
        "y_prob": p_eval
    }

In [21]:
# ============================================================
# 12. Training function
# ============================================================
def train_gnn(
    data,
    hidden_channels=64,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=200,
    patience=30,
    threshold=0.5
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = data.to(device)

    model = GraphSAGE(
        in_channels=data.num_node_features,
        hidden_channels=hidden_channels,
        dropout=0.3
    ).to(device)

    # Handle class imbalance
    y_train = data.y[data.train_mask]
    num_pos = (y_train == 1).sum().item()
    num_neg = (y_train == 0).sum().item()

    class_weights = torch.tensor(
        [1.0, num_neg / max(num_pos, 1)],
        dtype=torch.float32,
        device=device
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val_roc = -np.inf
    wait = 0

    for epoch in range(1, epochs + 1):
        # ---- train ----
        model.train()
        optimizer.zero_grad()

        logits = model(data.x, data.edge_index)
        loss = criterion(logits[data.train_mask], data.y[data.train_mask])

        loss.backward()
        optimizer.step()

        # ---- validate ----
        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index)
            val_metrics = evaluate_split(logits, data.y, data.val_mask, threshold=threshold)
            val_roc = val_metrics["roc_auc"]

        if val_roc > best_val_roc:
            best_val_roc = val_roc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % 20 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Loss {loss.item():.4f} | "
                f"Val ROC-AUC {val_metrics['roc_auc']:.4f} | "
                f"Val PR-AUC {val_metrics['pr_auc']:.4f}"
            )

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    # Load best model
    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        logits = model(data.x, data.edge_index)

    train_metrics = evaluate_split(logits, data.y, data.train_mask, threshold=threshold)
    val_metrics = evaluate_split(logits, data.y, data.val_mask, threshold=threshold)
    test_metrics = evaluate_split(logits, data.y, data.test_mask, threshold=threshold)

    print("\nBest model performance:")
    print(f"Train ROC-AUC: {train_metrics['roc_auc']:.4f} | Train PR-AUC: {train_metrics['pr_auc']:.4f}")
    print(f"Val   ROC-AUC: {val_metrics['roc_auc']:.4f} | Val   PR-AUC: {val_metrics['pr_auc']:.4f}")
    print(f"Test  ROC-AUC: {test_metrics['roc_auc']:.4f} | Test  PR-AUC: {test_metrics['pr_auc']:.4f}")

    print("\nTest classification report:")
    print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], digits=4))

    return {
        "model": model,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics
    }



In [22]:
# ============================================================
# 13. Train GNN V1
# ============================================================
print("\n" + "=" * 60)
print("Training GNN V1 (network + tabular)")
print("=" * 60)

results_v1 = train_gnn(
    data=data_v1,
    hidden_channels=64,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=200,
    patience=30,
    threshold=0.5
)




Training GNN V1 (network + tabular)
Epoch 001 | Loss 0.7134 | Val ROC-AUC 0.5246 | Val PR-AUC 0.2007
Epoch 020 | Loss 0.5832 | Val ROC-AUC 0.7814 | Val PR-AUC 0.5187
Epoch 040 | Loss 0.5400 | Val ROC-AUC 0.8226 | Val PR-AUC 0.5670
Epoch 060 | Loss 0.5102 | Val ROC-AUC 0.8333 | Val PR-AUC 0.5987
Epoch 080 | Loss 0.4946 | Val ROC-AUC 0.8365 | Val PR-AUC 0.6154
Epoch 100 | Loss 0.4699 | Val ROC-AUC 0.8381 | Val PR-AUC 0.6199
Epoch 120 | Loss 0.4651 | Val ROC-AUC 0.8391 | Val PR-AUC 0.6286
Early stopping at epoch 127

Best model performance:
Train ROC-AUC: 0.8661 | Train PR-AUC: 0.6286
Val   ROC-AUC: 0.8398 | Val   PR-AUC: 0.6233
Test  ROC-AUC: 0.8015 | Test  PR-AUC: 0.5141

Test classification report:
              precision    recall  f1-score   support

           0     0.9213    0.7482    0.8258      1096
           1     0.3566    0.6861    0.4693       223

    accuracy                         0.7377      1319
   macro avg     0.6390    0.7171    0.6476      1319
weighted avg     0.

In [23]:
# ============================================================
# 14. Train GNN V2
# ============================================================
print("\n" + "=" * 60)
print("Training GNN V2 (network + tabular + edu/job)")
print("=" * 60)

results_v2 = train_gnn(
    data=data_v2,
    hidden_channels=64,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=200,
    patience=30,
    threshold=0.5
)


Training GNN V2 (network + tabular + edu/job)
Epoch 001 | Loss 0.7009 | Val ROC-AUC 0.6333 | Val PR-AUC 0.2934
Epoch 020 | Loss 0.5778 | Val ROC-AUC 0.7704 | Val PR-AUC 0.5197
Epoch 040 | Loss 0.5343 | Val ROC-AUC 0.8182 | Val PR-AUC 0.5780
Epoch 060 | Loss 0.4948 | Val ROC-AUC 0.8323 | Val PR-AUC 0.6001
Epoch 080 | Loss 0.4810 | Val ROC-AUC 0.8372 | Val PR-AUC 0.6117
Epoch 100 | Loss 0.4657 | Val ROC-AUC 0.8383 | Val PR-AUC 0.6181
Early stopping at epoch 119

Best model performance:
Train ROC-AUC: 0.8660 | Train PR-AUC: 0.6327
Val   ROC-AUC: 0.8391 | Val   PR-AUC: 0.6174
Test  ROC-AUC: 0.7960 | Test  PR-AUC: 0.5137

Test classification report:
              precision    recall  f1-score   support

           0     0.9186    0.7418    0.8208      1096
           1     0.3479    0.6771    0.4597       223

    accuracy                         0.7309      1319
   macro avg     0.6333    0.7095    0.6402      1319
weighted avg     0.8222    0.7309    0.7597      1319



In [24]:
# ============================================================
# 15. Final comparison table
# ============================================================
summary = pd.DataFrame({
    "Model": ["GNN V1", "GNN V2"],
    "Val ROC-AUC": [
        results_v1["val_metrics"]["roc_auc"],
        results_v2["val_metrics"]["roc_auc"]
    ],
    "Val PR-AUC": [
        results_v1["val_metrics"]["pr_auc"],
        results_v2["val_metrics"]["pr_auc"]
    ],
    "Test ROC-AUC": [
        results_v1["test_metrics"]["roc_auc"],
        results_v2["test_metrics"]["roc_auc"]
    ],
    "Test PR-AUC": [
        results_v1["test_metrics"]["pr_auc"],
        results_v2["test_metrics"]["pr_auc"]
    ],
})

print("\nFinal summary:")
print(summary)


Final summary:
    Model  Val ROC-AUC  Val PR-AUC  Test ROC-AUC  Test PR-AUC
0  GNN V1     0.839806    0.623300      0.801453     0.514109
1  GNN V2     0.839124    0.617445      0.796040     0.513729
